### Downloading Train, Validation, and Test Datasets from Databricks Volume
This notebook automatically downloads the exported parquet files from the Databricks Volume `/Volumes/revenue_operations/gold/ml_datasets/` into `data/external/` and loads them into pandas DataFrames for local ML training.

In [1]:
import os
from pathlib import Path
import pandas as pd
from databricks.sdk import WorkspaceClient

# 1. Initialize Databricks Workspace Client (uses ~/.databrickscfg)
w = WorkspaceClient()

# 2. Configure paths
VOLUME_BASE = "/Volumes/revenue_operations/gold/ml_datasets"
datasets = {
    "train": f"{VOLUME_BASE}/delivery_risk_train",
    "val": f"{VOLUME_BASE}/delivery_risk_val",
    "test": f"{VOLUME_BASE}/delivery_risk_test"
}

# Define local target directory inside data/external
PROJECT_ROOT = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
EXTERNAL_DATA_DIR = PROJECT_ROOT / "data" / "external"
EXTERNAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Local target directory: {EXTERNAL_DATA_DIR}")

# 3. Download datasets from Databricks Volume
def download_and_load_volume(client, volume_path, target_dir):
    target_dir.mkdir(parents=True, exist_ok=True)
    items = list(client.files.list_directory_contents(volume_path))
    parquet_files = [f for f in items if f.path and f.path.endswith('.parquet')]
    
    for f in parquet_files:
        filename = os.path.basename(f.path)
        dest = target_dir / filename
        if not dest.exists():
            print(f"Downloading {filename} ({f.file_size / (1024*1024):.2f} MB)...")
            resp = client.files.download(f.path)
            with open(dest, "wb") as out:
                out.write(resp.contents.read())
        else:
            print(f"Using cached {filename}")
            
    # Read parquet files into DataFrame
    local_parquets = list(target_dir.glob("*.parquet"))
    return pd.concat([pd.read_parquet(p, engine="fastparquet") for p in local_parquets], ignore_index=True)

# 4. Download and load each split
print("\n--- Fetching Datasets ---")
train_df = download_and_load_volume(w, datasets["train"], EXTERNAL_DATA_DIR / "train")
val_df = download_and_load_volume(w, datasets["val"], EXTERNAL_DATA_DIR / "val")
test_df = download_and_load_volume(w, datasets["test"], EXTERNAL_DATA_DIR / "test")

print("\n--- Datasets Loaded Successfully ---")
print(f"Train shape:      {train_df.shape}")
print(f"Validation shape: {val_df.shape}")
print(f"Test shape:       {test_df.shape}")


Local target directory: C:\Users\prajw\Desktop\Projects\revenue_operations\data\external

--- Fetching Datasets ---
Using cached part-00000-tid-8739586932655165889-bfc25769-100d-4a45-aaf0-c1b59ce20e22-129-1.c000.snappy.parquet
Using cached part-00000-tid-1295487921349982676-09b09668-74ae-4d2b-8251-ee7d6332234b-130-1.c000.snappy.parquet
Using cached part-00000-tid-4112220302302538828-79f378c5-5462-424e-aed9-01da5478c7c4-131-1.c000.snappy.parquet

--- Datasets Loaded Successfully ---
Train shape:      (64238, 48)
Validation shape: (13579, 48)
Test shape:       (12141, 48)
